<div class='alert alert-warning'>

SciPy's interactive examples with Jupyterlite are experimental and may not always work as expected. Execution of cells containing imports may result in large downloads (up to 60MB of content for the first import from SciPy). Load times when importing from SciPy may take roughly 10-20 seconds. If you notice any problems, feel free to open an [issue](https://github.com/scipy/scipy/issues/new/choose).

</div>

The derivative can be used for finding the local extrema of a curve:


In [ ]:
import numpy as np
from scipy.interpolate import make_interp_spline, PPoly
x = np.linspace(0, 10, 70)
y = np.sin(x)
spl = make_interp_spline(x, y, k=4)  # spl is a quartic (degree 4) spline

Now, differentiate the spline and find the zeros of the derivative.
`BSpline` objects do not have a ``roots`` method, so we convert the
derivative to a `PPoly`, which represents the same curve as a piecewise
polynomial. We fit a spline of degree 4 rather than the default cubic,
so that its derivative is smooth enough to locate the extrema
accurately: the derivative of a quartic spline is a cubic spline with
the same breakpoints, and `PPoly.roots` finds all real roots of each
cubic piece directly, without iterating.


In [ ]:
dspl = spl.derivative()
xe = PPoly.from_spline(dspl).roots(extrapolate=False)
xe / np.pi

array([0.50000001, 1.5       , 2.49999998])

The array holds the locations, in units of $\pi$, where the
derivative vanishes on the data interval $[0, 10]$. This agrees
well with roots $\pi/2 + n\pi$ of $\cos(x) = \sin'(x)$.

A zero of the derivative only tells us that the curve is flat there,
which usually means a local extremum. To tell maxima from minima, check
the sign of the second derivative: negative at a maximum, positive at a
minimum.


In [ ]:
np.sign(spl.derivative(2)(xe))

array([-1.,  1., -1.])

So the curve has maxima at $x \approx \pi/2$ and
$x \approx 5\pi/2$, and a minimum at $x \approx 3\pi/2$.

The coefficient array of the returned spline is padded with ``k + 1``
extra trailing entries, which are not used in evaluations, as the
examples below demonstrate:


In [ ]:
from scipy.interpolate import BSpline, make_interp_spline
x = np.arange(11)
y = x**4 + np.sin(x)
spl = make_interp_spline(x, y, k=3)  # spl is a cubic (degree 3) spline

A spline of degree ``k`` is built from polynomial pieces of degree
``k``, joined at the knots ``t``. The number of knots, the number of
coefficients, and the degree are related by
``len(t) - len(c) == k + 1``. Padded coefficients instead satisfy
``len(c) == len(t)``:


In [ ]:
len(spl.t), len(spl.c), spl.k    # knots, coefficients, degree of spl

(15, 11, 3)

In [ ]:
b = spl.derivative()             # b is a quadratic spline
len(b.t), len(b.c), b.k          # knots, coefficients, degree of b

(13, 13, 2)

In [ ]:
b.c[len(b.t) - b.k - 1:]         # zero padding, unused in evaluations

array([0., 0., 0.])

Slice off the padding to get exactly ``len(b.t) - b.k - 1`` coefficients:


In [ ]:
c = b.c[:len(b.t) - b.k - 1]
len(c)

10

The trimmed coefficients define the same spline, since the padding
never enters evaluations. Evaluating the spline before (``b``) and
after (``BSpline(b.t, c, b.k)``) trimming on a grid of points spanning
the data range gives the same values:


In [ ]:
xx = np.linspace(0, 10, 51)
np.allclose(BSpline(b.t, c, b.k)(xx), b(xx))

True